# 1. Configuração do Ambiente e Ingestão de Dados

Nesta etapa, importamos as bibliotecas matemáticas e estruturais e montamos o diretório do Google Drive. O carregamento do arquivo Excel é feito de forma interativa.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive, files

# Monta o drive para permitir leitura/escrita em nuvem
drive.mount('/content/drive')

# 1. Upload interativo do arquivo
print("Selecione o seu arquivo .xlsx:")
uploaded = files.upload()
nome_arquivo_entrada = list(uploaded.keys())[0]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Selecione o seu arquivo .xlsx:


# 2. Processamento e Interpolação (Grid 15cm)

Esta é a função principal, contendo diversas travas de segurança arquitetônicas:
* **Prevenção de Travamento:** Caso as colunas de "Poço" ou "Profundidade" não sejam identificadas, o código lança um erro explícito em vez de aguardar um `input()` infinito.
* **Prevenção de Falhas no Algoritmo:** Valores `NaN` na coluna de profundidade são descartados e profundidades duplicadas são consolidadas (a média é aplicada aos números e o primeiro valor às strings), garantindo a ordenação estrita exigida pelo `merge_asof`.
* **Tratamento Híbrido de Dados:** O script isola colunas puramente numéricas de colunas categóricas (como códigos de fácies). O valor `-9999` é imputado apenas nas variáveis contínuas, enquanto as categóricas recebem `"Sem_Dado"`, preservando o tipo de dado original da matriz para a etapa de detecção de anomalias.
* **Tolerância Matemática:** Foi aplicada uma correção de ponto flutuante, arredondando a profundidade original e adicionando um delta (0.001) na margem de busca.

In [ ]:
def processar_geochem_15cm_seguro(caminho_arquivo, step=0.15):
    # Carregar dados
    df = pd.read_excel(caminho_arquivo)

    # Identificar colunas
    col_poco = next((c for c in df.columns if c.upper() in ['POCO', 'WELL', 'POÇO', 'ID_POCO']), None)
    col_prof = next((c for c in df.columns if c.upper() in ['PROFUNDIDADE', 'DEPTH', 'PROF', 'METRAGEM']), None)

    # TRAVA 1: Evita o congelamento do Colab substituindo input() por um erro claro
    if not col_poco or not col_prof:
        raise ValueError(f"As colunas de Poço e Profundidade não foram detectadas. \nColunas disponíveis: {list(df.columns)}.\nPor favor, renomeie os cabeçalhos no arquivo de origem.")

    # TRAVA 2: Remove linhas onde a profundidade é nula (quebra o merge)
    df = df.dropna(subset=[col_prof])

    # TRAVA 3: Força a profundidade para numérico e arredonda para equalizar com o grid
    df[col_prof] = pd.to_numeric(df[col_prof], errors='coerce')
    df[col_prof] = df[col_prof].round(2)

    # Separa variáveis para evitar corrupção de tipos (ex: códigos litológicos virando floats)
    colunas_dados = [c for c in df.columns if c not in [col_poco, col_prof]]
    col_numericas = df[colunas_dados].select_dtypes(include=[np.number]).columns.tolist()
    col_categoricas = df[colunas_dados].select_dtypes(exclude=[np.number]).columns.tolist()

    # TRAVA 4: Resolve duplicatas na mesma profundidade agregando pela média (num) e mantendo o 1º valor (cat)
    agg_dict = {col: 'mean' for col in col_numericas}
    agg_dict.update({col: 'first' for col in col_categoricas})
    df = df.groupby([col_poco, col_prof], as_index=False).agg(agg_dict)

    # Ordenação estrita obrigatória para o merge_asof
    df = df.sort_values(by=[col_poco, col_prof])

    lista_final = []

    # TRAVA 5: Adiciona um limite microscópico para capturar valores limítrofes flutuantes
    tolerance_ajustada = (step / 2) + 0.001

    # Processar cada poço individualmente
    for nome_poco, group in df.groupby(col_poco):
        # Criar o grid contínuo
        p_min = np.floor(group[col_prof].min() / step) * step
        p_max = np.ceil(group[col_prof].max() / step) * step

        grid_profs = np.arange(p_min, p_max + (step/2), step)
        df_grid = pd.DataFrame({col_prof: grid_profs})
        df_grid[col_prof] = df_grid[col_prof].round(2)

        # Alinhamento pelo vizinho mais próximo
        df_resampled = pd.merge_asof(
            df_grid,
            group,
            on=col_prof,
            direction='nearest',
            tolerance=tolerance_ajustada
        )

        df_resampled[col_poco] = nome_poco

        # TRAVA 6: Preenchimento segregado
        if col_numericas:
            df_resampled[col_numericas] = df_resampled[col_numericas].fillna(-9999)
        if col_categoricas:
            df_resampled[col_categoricas] = df_resampled[col_categoricas].fillna("Sem_Dado")

        lista_final.append(df_resampled)

    return pd.concat(lista_final, ignore_index=True)

# 3. Execução e Download do Resultado

Acionamos a função passando o arquivo carregado no início. O dataframe gerado é então consolidado em um novo arquivo Excel `.xlsx` e baixado automaticamente no ambiente local, livre de distorções de amostragem.

In [ ]:
# 2. Executar processamento
print("Iniciando regularização do grid (15cm) com verificações de integridade...")
df_resultado = processar_geochem_15cm_seguro(nome_arquivo_entrada)

# 3. Salvar e baixar
nome_saida = 'geoquimica_15cm_regularizado.xlsx'
df_resultado.to_excel(nome_saida, index=False)
print(f"Processamento concluído com sucesso! Gerando arquivo: {nome_saida}")
files.download(nome_saida)